# Analyse Statistique avec SciPy
## Exercices 1 a 8 — Statistiques, Tests d'Hypotheses, Regression, ANOVA




## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy
from scipy import stats
from scipy.stats import (
    norm, binom,
    ttest_ind,
    f_oneway,
    pearsonr, spearmanr,
    linregress,
    describe
)

print(f"SciPy version  : {scipy.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")


## Exercice 1 — Utilisation de base de SciPy

**Objectif** : importer SciPy et explorer les sous-modules disponibles.


In [ ]:
# Version et sous-modules principaux de SciPy
print("Version SciPy :", scipy.__version__)
print("\nSous-modules principaux :")
submodules = [
    ("scipy.stats",       "Statistiques et distributions de probabilite"),
    ("scipy.optimize",    "Optimisation et recherche de racines"),
    ("scipy.linalg",      "Algebre lineaire"),
    ("scipy.integrate",   "Integration numerique"),
    ("scipy.interpolate", "Interpolation"),
    ("scipy.signal",      "Traitement du signal"),
    ("scipy.fft",         "Transformee de Fourier rapide"),
]
for name, desc in submodules:
    print(f"  {name:<25} -> {desc}")


## Exercice 2 — Statistiques Descriptives

**Objectif** : calculer la moyenne, la mediane, la variance et l'ecart-type
avec SciPy/NumPy sur un echantillon de donnees.


In [ ]:
data = [12, 15, 13, 12, 18, 20, 22, 21]

# --- Calculs statistiques ---
mean    = np.mean(data)
median  = np.median(data)
var     = np.var(data, ddof=1)    # ddof=1 -> variance d'echantillon (non biaisee)
std_dev = np.std(data, ddof=1)
minimum = np.min(data)
maximum = np.max(data)
q1, q3  = np.percentile(data, [25, 75])
iqr     = q3 - q1

# scipy.stats.describe donne un resume complet
desc = describe(data)

print("Statistiques descriptives de l'echantillon")
print("=" * 45)
print(f"  N           : {desc.nobs}")
print(f"  Minimum     : {minimum}")
print(f"  Maximum     : {maximum}")
print(f"  Etendue     : {maximum - minimum}")
print(f"  Moyenne     : {mean:.4f}")
print(f"  Mediane     : {median:.4f}")
print(f"  Variance    : {var:.4f}  (ddof=1)")
print(f"  Ecart-type  : {std_dev:.4f}  (ddof=1)")
print(f"  Q1          : {q1:.2f}")
print(f"  Q3          : {q3:.2f}")
print(f"  IQR         : {iqr:.2f}")
print(f"  Asymetrie   : {desc.skewness:.4f}")
print(f"  Kurtosis    : {desc.kurtosis:.4f}")


In [ ]:
# Visualisation : histogramme + boxplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(data, bins=6, color='#4FC3F7', edgecolor='white', alpha=0.85)
ax1.axvline(mean,   color='#FF8A65', lw=2, linestyle='--', label=f'Moyenne = {mean:.2f}')
ax1.axvline(median, color='#81C784', lw=2, linestyle=':',  label=f'Mediane = {median:.2f}')
ax1.set_title('Distribution de l\'echantillon', fontweight='bold')
ax1.set_xlabel('Valeur'); ax1.set_ylabel('Frequence')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.boxplot(data, patch_artist=True,
            boxprops=dict(facecolor='#4FC3F7', alpha=0.7),
            medianprops=dict(color='#FF8A65', lw=2))
ax2.set_title('Boite a moustaches', fontweight='bold')
ax2.set_ylabel('Valeur'); ax2.grid(alpha=0.3)

plt.suptitle('Exercice 2 — Statistiques Descriptives', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Exercice 3 — Distribution Normale

**Objectif** : generer et visualiser une distribution normale
avec moyenne = 50 et ecart-type = 10.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

mu    = 50   # moyenne
sigma = 10   # ecart-type

# Espace de valeurs pour la courbe theorique
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)

# Densite de probabilite (PDF) et fonction de repartition (CDF)
pdf = norm.pdf(x, loc=mu, scale=sigma)
cdf = norm.cdf(x, loc=mu, scale=sigma)

# Echantillon aleatoire
np.random.seed(42)
sample = norm.rvs(loc=mu, scale=sigma, size=1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- PDF + histogramme ---
axes[0].hist(sample, bins=30, density=True,
             color='#4FC3F7', edgecolor='white', alpha=0.6, label='Echantillon (n=1000)')
axes[0].plot(x, pdf, color='#FF8A65', lw=2.5, label='PDF theorique')
# Zones : mu +/- 1, 2, 3 sigma
for k, alpha_fill in [(3, 0.10), (2, 0.18), (1, 0.30)]:
    axes[0].fill_between(x, pdf,
                         where=(x >= mu - k*sigma) & (x <= mu + k*sigma),
                         color='#FF8A65', alpha=alpha_fill,
                         label=f'±{k}σ' if k <= 2 else None)
axes[0].axvline(mu, color='white', lw=1.5, linestyle='--', label=f'μ = {mu}')
axes[0].set_title('PDF — Distribution Normale', fontweight='bold')
axes[0].set_xlabel('Valeur'); axes[0].set_ylabel('Densite')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# --- CDF ---
axes[1].plot(x, cdf, color='#81C784', lw=2.5)
axes[1].axhline(0.5, color='white', lw=1, linestyle=':', alpha=0.6)
axes[1].axvline(mu,  color='white', lw=1, linestyle=':', alpha=0.6)
axes[1].set_title('CDF — Fonction de Repartition', fontweight='bold')
axes[1].set_xlabel('Valeur'); axes[1].set_ylabel('P(X <= x)')
axes[1].grid(alpha=0.3)

# --- Q-Q plot ---
from scipy.stats import probplot
probplot(sample, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot (normalite)', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Exercice 3 — Distribution Normale (μ={mu}, σ={sigma})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Probabilites cles
print(f"P(X <= 50) = {norm.cdf(mu, mu, sigma):.4f}  (doit etre 0.5000)")
print(f"P(40 <= X <= 60) = {norm.cdf(60,mu,sigma) - norm.cdf(40,mu,sigma):.4f}  (regle des 68%)")
print(f"P(30 <= X <= 70) = {norm.cdf(70,mu,sigma) - norm.cdf(30,mu,sigma):.4f}  (regle des 95%)")
print(f"P(20 <= X <= 80) = {norm.cdf(80,mu,sigma) - norm.cdf(20,mu,sigma):.4f}  (regle des 99.7%)")


## Exercice 4 — Test T de Student




In [ ]:
np.random.seed(42)
data1 = np.random.normal(50, 10, 100)   # groupe 1 : mu=50
data2 = np.random.normal(60, 10, 100)   # groupe 2 : mu=60

# Test T bilateral (two-sided)
t_stat, p_value = ttest_ind(data1, data2)

alpha = 0.05
print("Test T de Student — deux echantillons independants")
print("=" * 50)
print(f"  Moyenne data1  : {data1.mean():.4f}")
print(f"  Moyenne data2  : {data2.mean():.4f}")
print(f"  Difference     : {data2.mean() - data1.mean():.4f}")
print(f"  T-statistique  : {t_stat:.4f}")
print(f"  P-value        : {p_value:.2e}")
print(f"  Seuil alpha    : {alpha}")
print()
if p_value < alpha:
    print(f"  Conclusion : p ({p_value:.2e}) < alpha ({alpha})")
    print("  -> On REJETTE H0 : les moyennes sont significativement differentes.")
else:
    print(f"  Conclusion : p ({p_value:.2e}) >= alpha ({alpha})")
    print("  -> On NE REJETTE PAS H0 : pas de difference significative.")


In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogrammes superposes
axes[0].hist(data1, bins=20, alpha=0.65, color='#4FC3F7',
             edgecolor='white', label=f'data1  (μ={data1.mean():.1f})')
axes[0].hist(data2, bins=20, alpha=0.65, color='#FF8A65',
             edgecolor='white', label=f'data2  (μ={data2.mean():.1f})')
axes[0].axvline(data1.mean(), color='#4FC3F7', lw=2, linestyle='--')
axes[0].axvline(data2.mean(), color='#FF8A65', lw=2, linestyle='--')
axes[0].set_title('Distribution des deux groupes', fontweight='bold')
axes[0].set_xlabel('Valeur'); axes[0].set_ylabel('Frequence')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Boites a moustaches
bp = axes[1].boxplot([data1, data2], patch_artist=True, notch=True,
                     medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], ['#4FC3F7','#FF8A65']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_xticks([1,2]); axes[1].set_xticklabels(['data1 (μ=50)','data2 (μ=60)'])
axes[1].set_title(f'Comparaison — T={t_stat:.2f}, p={p_value:.2e}', fontweight='bold')
axes[1].set_ylabel('Valeur'); axes[1].grid(alpha=0.3)

plt.suptitle('Exercice 4 — Test T de Student', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Exercice 5 — Regression Lineaire




In [ ]:
house_sizes  = np.array([50, 70, 80, 100, 120])    # en m2
house_prices = np.array([150000, 200000, 210000, 250000, 280000])  # en USD

# Regression lineaire avec scipy.stats.linregress
slope, intercept, r_value, p_value, std_err = linregress(house_sizes, house_prices)

r_squared   = r_value ** 2
pred_90     = slope * 90 + intercept

print("Regression Lineaire : prix = pente x surface + constante")
print("=" * 55)
print(f"  Pente (slope)     : {slope:.2f} USD par m2")
print(f"  Constante         : {intercept:.2f} USD")
print(f"  R (correlation)   : {r_value:.4f}")
print(f"  R2 (determination): {r_squared:.4f}  ({r_squared*100:.2f}% de variance expliquee)")
print(f"  P-value (pente)   : {p_value:.6f}")
print(f"  Erreur standard   : {std_err:.4f}")
print()
print(f"  Equation          : Prix = {slope:.2f} x Surface + {intercept:.2f}")
print()
print(f"  Prediction 90 m2  : {pred_90:,.0f} USD")
print()
print("Interpretation de la pente :")
print(f"  Chaque m2 supplementaire augmente le prix de {slope:.0f} USD en moyenne.")


In [ ]:
# Visualisation regression
x_range   = np.linspace(40, 130, 200)
y_pred    = slope * x_range + intercept
residuals = house_prices - (slope * house_sizes + intercept)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Droite de regression
axes[0].scatter(house_sizes, house_prices, color='#FF8A65', s=100, zorder=5, label='Donnees')
axes[0].plot(x_range, y_pred, color='#4FC3F7', lw=2, label=f'y = {slope:.1f}x + {intercept:.0f}')
axes[0].scatter([90], [pred_90], color='#81C784', s=150, marker='*', zorder=6,
                label=f'Prediction 90m2 = {pred_90:,.0f}')
# Residus
for xi, yi, yp in zip(house_sizes, house_prices, slope*house_sizes+intercept):
    axes[0].plot([xi, xi], [yi, yp], color='white', lw=1, linestyle=':', alpha=0.6)
axes[0].set_title(f'Regression Lineaire (R2={r_squared:.4f})', fontweight='bold')
axes[0].set_xlabel('Surface (m2)'); axes[0].set_ylabel('Prix (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Residus
axes[1].bar(range(len(residuals)), residuals, color=['#81C784' if r>=0 else '#FF8A65' for r in residuals],
            edgecolor='white', alpha=0.85)
axes[1].axhline(0, color='white', lw=1.5, linestyle='--')
axes[1].set_xticks(range(len(house_sizes)))
axes[1].set_xticklabels([f'{s}m2' for s in house_sizes])
axes[1].set_title('Residus (erreurs)', fontweight='bold')
axes[1].set_ylabel('Residu (USD)'); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Exercice 5 — Regression Lineaire (Prix immobilier)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Exercice 6 — Analyse de la Variance (ANOVA)




In [ ]:
fertilizer_1 = [5, 6, 7, 6, 5]
fertilizer_2 = [7, 8, 7, 9, 8]
fertilizer_3 = [4, 5, 4, 3, 4]

F_stat, p_anova = f_oneway(fertilizer_1, fertilizer_2, fertilizer_3)

alpha = 0.05
print("ANOVA a un facteur — Effet des engrais sur la croissance")
print("=" * 57)
print(f"  Engrais 1 : moyenne={np.mean(fertilizer_1):.2f}, std={np.std(fertilizer_1, ddof=1):.2f}")
print(f"  Engrais 2 : moyenne={np.mean(fertilizer_2):.2f}, std={np.std(fertilizer_2, ddof=1):.2f}")
print(f"  Engrais 3 : moyenne={np.mean(fertilizer_3):.2f}, std={np.std(fertilizer_3, ddof=1):.2f}")
print()
print(f"  F-statistique : {F_stat:.4f}")
print(f"  P-value       : {p_anova:.6f}")
print(f"  Seuil alpha   : {alpha}")
print()
if p_anova < alpha:
    print(f"  Conclusion : p ({p_anova:.6f}) < alpha ({alpha})")
    print("  -> On REJETTE H0 : les engrais ont des effets SIGNIFICATIVEMENT differents.")
else:
    print(f"  Conclusion : p ({p_anova:.6f}) >= alpha ({alpha})")
    print("  -> On NE REJETTE PAS H0.")

print()
print("  Si p > 0.05 :")
print("  On ne rejetterait pas H0 : aucune difference significative entre les engrais.")
print("  Cela signifierait que les variations observees sont attribuables au hasard.")
print("  On ne pourrait pas conclure qu'un engrais est superieur aux autres.")


In [ ]:
# Visualisation ANOVA
groups       = [fertilizer_1, fertilizer_2, fertilizer_3]
group_labels = ['Engrais 1', 'Engrais 2', 'Engrais 3']
group_colors = ['#4FC3F7', '#81C784', '#FF8A65']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Boxplot
bp = axes[0].boxplot(groups, patch_artist=True, notch=False,
                     medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], group_colors):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[0].set_xticks([1,2,3]); axes[0].set_xticklabels(group_labels)
axes[0].set_title(f'Boxplot par groupe
F={F_stat:.2f}, p={p_anova:.4f}', fontweight='bold')
axes[0].set_ylabel('Croissance (cm)'); axes[0].grid(alpha=0.3)

# Barres avec ecart-type
means  = [np.mean(g) for g in groups]
errors = [np.std(g, ddof=1) for g in groups]
bars   = axes[1].bar(group_labels, means, color=group_colors,
                     edgecolor='white', alpha=0.85, yerr=errors,
                     capsize=6, error_kw=dict(color='white', lw=2))
axes[1].axhline(np.mean([v for g in groups for v in g]),
                color='white', lw=1.5, linestyle='--', label='Moyenne globale')
axes[1].set_title('Moyenne ± ecart-type', fontweight='bold')
axes[1].set_ylabel('Croissance (cm)')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15,
                 f'{bar.get_height():.1f}', ha='center', color='white', fontsize=9)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis='y')

# Donnees brutes stripplot
for i, (grp, label, color) in enumerate(zip(groups, group_labels, group_colors), 1):
    jitter = np.random.uniform(-0.08, 0.08, len(grp))
    axes[2].scatter([i + j for j in jitter], grp,
                    color=color, s=80, alpha=0.9, edgecolors='white', lw=0.5)
    axes[2].hlines(np.mean(grp), i-0.25, i+0.25, colors='white', lw=2)
axes[2].set_xticks([1,2,3]); axes[2].set_xticklabels(group_labels)
axes[2].set_title('Donnees individuelles + moyenne', fontweight='bold')
axes[2].set_ylabel('Croissance (cm)'); axes[2].grid(alpha=0.3)

plt.suptitle('Exercice 6 — ANOVA : Effet des engrais', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Exercice 7 (Optionnel) — Distribution Binomiale

**Objectif** : calculer les probabilites associees a une distribution binomiale.

**Exemple** : lancer une piece de monnaie 10 fois (p=0.5). Calculer
la probabilite d'obtenir exactement 5 faces, puis tracer la distribution complete.


In [ ]:
n = 10    # nombre d'essais
p = 0.5   # probabilite de succes

# Probabilite exacte : P(X = 5)
p_exact_5 = binom.pmf(5, n, p)
print(f"P(X = 5 | n=10, p=0.5)  = {p_exact_5:.4f}  ({p_exact_5*100:.2f}%)")

# Distribution complete
k_vals = np.arange(0, n+1)
pmf    = binom.pmf(k_vals, n, p)
cdf    = binom.cdf(k_vals, n, p)

# Probabilites cumulees
print(f"P(X <= 5)               = {binom.cdf(5, n, p):.4f}")
print(f"P(X >= 5)               = {1 - binom.cdf(4, n, p):.4f}")
print(f"P(3 <= X <= 7)          = {binom.cdf(7,n,p) - binom.cdf(2,n,p):.4f}")
print()
print("Distribution complete :")
print(f"  {'k':>3}  {'P(X=k)':>8}  {'P(X<=k)':>9}")
print("  " + "-"*25)
for k, prob, cum in zip(k_vals, pmf, cdf):
    marker = "  <-- P(X=5)" if k == 5 else ""
    print(f"  {k:>3}  {prob:>8.4f}  {cum:>9.4f}{marker}")


In [ ]:
# Visualisation distribution binomiale
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PMF
bar_colors = ['#FF8A65' if k == 5 else '#4FC3F7' for k in k_vals]
axes[0].bar(k_vals, pmf, color=bar_colors, edgecolor='white', alpha=0.85)
axes[0].bar(5, pmf[5], color='#FF8A65', edgecolor='white', alpha=0.95,
            label=f'P(X=5) = {p_exact_5:.4f}')
axes[0].set_title('PMF — Loi Binomiale B(10, 0.5)', fontweight='bold')
axes[0].set_xlabel('Nombre de succes (k)')
axes[0].set_ylabel('P(X = k)')
axes[0].set_xticks(k_vals)
axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

# CDF
axes[1].step(k_vals, cdf, color='#81C784', lw=2.5, where='post')
axes[1].scatter(k_vals, cdf, color='#81C784', s=50, zorder=5)
axes[1].axhline(0.5, color='white', lw=1, linestyle=':', alpha=0.6)
axes[1].set_title('CDF — Fonction de Repartition', fontweight='bold')
axes[1].set_xlabel('k'); axes[1].set_ylabel('P(X <= k)')
axes[1].set_xticks(k_vals); axes[1].grid(alpha=0.3)

plt.suptitle('Exercice 7 — Distribution Binomiale B(n=10, p=0.5)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Exercice 8 (Optionnel) — Coefficients de Correlation




In [ ]:
data = pd.DataFrame({
    'age':    [23, 25, 30, 35, 40],
    'income': [35000, 40000, 50000, 60000, 70000]
})

# Correlation de Pearson
r_pearson,  p_pearson  = pearsonr(data['age'], data['income'])
# Correlation de Spearman
r_spearman, p_spearman = spearmanr(data['age'], data['income'])

print("Correlations entre l'age et le revenu")
print("=" * 45)
print(f"  N               : {len(data)}")
print()
print(f"  Pearson  r      : {r_pearson:.4f}")
print(f"  Pearson  p-val  : {p_pearson:.4f}")
print(f"  -> {('Correlation significative' if p_pearson < 0.05 else 'Non significative')} (alpha=0.05)")
print()
print(f"  Spearman r      : {r_spearman:.4f}")
print(f"  Spearman p-val  : {p_spearman:.4f}")
print(f"  -> {('Correlation significative' if p_spearman < 0.05 else 'Non significative')} (alpha=0.05)")
print()
print("Interpretation :")
print(f"  Pearson  r={r_pearson:.2f} : relation lineaire quasi-parfaite entre age et revenu.")
print(f"  Spearman r={r_spearman:.2f} : relation monotone parfaite (rang identique).")
print("  Les deux coefficients convergent : plus l'age augmente, plus le revenu augmente.")


In [ ]:
# Visualisation correlations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scatter age vs income
axes[0].scatter(data['age'], data['income'], color='#FF8A65', s=120,
                edgecolors='white', lw=1.5, zorder=5)
# Droite de regression
slope_c, intercept_c, *_ = linregress(data['age'], data['income'])
x_c = np.linspace(20, 45, 100)
axes[0].plot(x_c, slope_c*x_c + intercept_c, color='#4FC3F7', lw=2,
             label=f'y = {slope_c:.0f}x + {intercept_c:.0f}')
# Annotations
for _, row in data.iterrows():
    axes[0].annotate(f"({row['age']}, {row['income']//1000}k)",
                     (row['age'], row['income']),
                     textcoords='offset points', xytext=(5,5), fontsize=8, color='white')
axes[0].set_title(f'Scatter : age vs revenu
Pearson r={r_pearson:.4f}', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Revenu (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v/1000:.0f}k'))
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Matrice de correlation (heatmap manuelle)
corr_matrix = data.corr()
im = axes[1].imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['age','income'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['age','income'])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{corr_matrix.values[i,j]:.4f}',
                     ha='center', va='center', color='black', fontweight='bold')
axes[1].set_title('Matrice de correlation (Pearson)', fontweight='bold')

# Comparaison Pearson vs Spearman
metrics = ['Pearson', 'Spearman']
values  = [r_pearson, r_spearman]
bars    = axes[2].bar(metrics, values, color=['#4FC3F7','#81C784'],
                      edgecolor='white', alpha=0.85, width=0.4)
axes[2].set_ylim(0, 1.15)
axes[2].axhline(0.7, color='#FFD54F', lw=1.5, linestyle='--', label='Seuil forte correlation (0.7)')
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x()+bar.get_width()/2, val+0.02,
                 f'r = {val:.4f}', ha='center', color='white', fontweight='bold')
axes[2].set_title('Comparaison des coefficients', fontweight='bold')
axes[2].set_ylabel('Coefficient de correlation')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3, axis='y')

plt.suptitle('Exercice 8 — Correlations Pearson & Spearman',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
